# SIGMOD Exp 2 Variant: Distinct Historical / Delta Targets

This variant keeps the original Exp2 notebook intact and changes only the sweep semantics:

1. Historical sweep increases the exact number of historical scans.
2. Historical probes are disabled.
3. Each historical scan targets a fresh readable timestamp when possible.
4. Delta sweep increases the exact number of delta transactions.
5. Each delta transaction targets a fresh readable-timestamp pair when possible.
6. SNAP rows are loaded from fixed values by default, but can be re-run later.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import importlib

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_READABLE_EVERY,
    apply_paper_style,
    current_run_stamp,
    display_name,
    ensure_dirs,
    run_checked,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_distinct_crossover').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['ivmh', 'heap', 'chain', 'par']
TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}

CONFIG = {
    'warehouse_count': SIGMOD_HTAP_WAREHOUSE_COUNT,
    'txn_count': SIGMOD_HTAP_TXN_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.0001,
    'probe_ratio': 0.01,
    'txn_gc_ratio': 0.05,
    'readable_every': SIGMOD_READABLE_EVERY,
    'repeat': 5,
    'trim': 1,
    'timeout_sec': 900,
    'rerun_snap': False,
}

SWEEP = {
    'history_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
    'delta_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
}

STYLE = {
    ('naive', ''): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', ''): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'Write Repair'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'Write Repair'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'Write Repair'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}

RUN_STAMP = current_run_stamp()

CONFIG_TAG = '_'.join([
    f"wc{CONFIG['warehouse_count']}",
    f"tc{CONFIG['txn_count']}",
    f"bn{CONFIG['bucket_num']}",
    f"ur{str(CONFIG['update_ratio']).replace('.', 'p')}",
    f"pr{str(CONFIG['probe_ratio']).replace('.', 'p')}",
    f"gc{str(CONFIG['txn_gc_ratio']).replace('.', 'p')}",
    f"re{CONFIG['readable_every']}",
    f"rep{CONFIG['repeat']}",
    'distinct',
])

BASE_ARGS = [
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--readable-every', str(CONFIG['readable_every']),
]

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)
print('CONFIG :', CONFIG)
print('STAMP  :', RUN_STAMP)
print('TAG    :', CONFIG_TAG)
print('SNAP   :', 'fixed rows' if not CONFIG['rerun_snap'] else 'rerun')


In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')


In [ ]:
FIXED_SNAP_DELTA_ROWS = [
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 40139.22504744444, 'total_ms': 401.3922504744444, 'delta_ratio': 0.00},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 41288.46775166667, 'total_ms': 412.8846775166667, 'delta_ratio': 0.01},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 41559.591691444446, 'total_ms': 415.5959169144445, 'delta_ratio': 0.02},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 42129.533485333326, 'total_ms': 421.29533485333326, 'delta_ratio': 0.04},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 42632.21794944445, 'total_ms': 426.3221794944445, 'delta_ratio': 0.06},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 42890.86272711111, 'total_ms': 428.90862727111113, 'delta_ratio': 0.08},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 42336.72021111111, 'total_ms': 423.3672021111111, 'delta_ratio': 0.10},
]

FIXED_SNAP_HISTORY_ROWS = [
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 40651.66512711111, 'total_ms': 406.51665127111113, 'history_ratio': 0.00},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 40687.81570233333, 'total_ms': 406.87815702333336, 'history_ratio': 0.01},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 39695.93911433333, 'total_ms': 396.9593911433333, 'history_ratio': 0.02},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 39469.83057122222, 'total_ms': 394.69830571222224, 'history_ratio': 0.04},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 39229.225423, 'total_ms': 392.29225423, 'history_ratio': 0.06},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 38858.987995888885, 'total_ms': 388.5898799588888, 'history_ratio': 0.08},
    {'table_type': 'naive', 'repair_type': '', 'duration_ms': 37854.40020022222, 'total_ms': 378.5440020022222, 'history_ratio': 0.10},
]

def merge_args(base, extra):
    merged = {}
    for args in (base, extra):
        it = iter(args)
        for token in it:
            merged[token] = next(it)
    out = []
    for k, v in merged.items():
        out.extend([k, v])
    return out


def trim_trial_runs(df):
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()


def collapse_repairs(df, table_type):
    if table_type in {'naive', 'ivmh'}:
        collapsed = df.groupby(['table_type'], as_index=False)['duration_ms'].mean()
        collapsed['repair_type'] = ''
        return collapsed[['table_type', 'repair_type', 'duration_ms']]
    return df


def run_single(extra_args, include_snap=False):
    table_types = ['naive'] + TABLE_TYPES if include_snap else TABLE_TYPES
    rows = []
    for table_type in table_types:
        trials = []
        print('  table=', table_type)
        for trial in range(CONFIG['repeat']):
            result = run_checked([str(BIN), *merge_args(BASE_ARGS, extra_args), '--table-type', table_type], ROOT, quiet=True, timeout=CONFIG['timeout_sec'])
            df = parse_result(result.stdout, table_type)
            if df.empty:
                raise RuntimeError(f'No parsed rows for {table_type}')
            df['tx_type'] = df['tx_type'].replace(TX_MAP)
            total = df.groupby('repair_type', as_index=False)['duration_ms'].sum()
            total['table_type'] = table_type
            total['trial'] = trial
            trials.append(total)
        df_all = trim_trial_runs(pd.concat(trials, ignore_index=True))
        df_avg = df_all.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].mean()
        rows.append(collapse_repairs(df_avg, table_type))
    out = pd.concat(rows, ignore_index=True)
    out['total_ms'] = out['duration_ms'] / CONFIG['txn_count']
    return out


def build_history_args(history_pct):
    history_scan_count = int(round(CONFIG['txn_count'] * history_pct))
    update_ratio = 0.20
    probe_ratio = 0.40
    delta_ratio = 0.00
    scan_ratio = 0.40
    actual_scan_count = round(CONFIG['txn_count'] * (1.0 - CONFIG['txn_gc_ratio']) * scan_ratio)
    scan_reuse_ratio = 0.0 if actual_scan_count == 0 else min(1.0, history_scan_count / actual_scan_count)
    return [
        '--txn-update-ratio', str(update_ratio),
        '--txn-probe-ratio', str(probe_ratio),
        '--txn-scan-ratio', str(scan_ratio),
        '--txn-delta-ratio', str(delta_ratio),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', str(scan_reuse_ratio),
        '--probe-history-ratio', '0.0',
        '--distinct-history-targets',
    ]


def build_delta_args(delta_pct):
    delta_ratio = float(delta_pct)
    update_ratio = 0.20
    scan_ratio = 0.40
    probe_ratio = 1.0 - update_ratio - scan_ratio - delta_ratio
    if probe_ratio < 0:
        raise ValueError(f'Invalid delta_pct {delta_pct}: negative probe share')
    return [
        '--txn-update-ratio', str(update_ratio),
        '--txn-probe-ratio', str(probe_ratio),
        '--txn-scan-ratio', str(scan_ratio),
        '--txn-delta-ratio', str(delta_ratio),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', '0.0',
        '--probe-history-ratio', '0.0',
        '--distinct-delta-targets',
    ]


def add_snap_rows(df, fixed_rows, x_col):
    snap_df = pd.DataFrame(fixed_rows)
    out = pd.concat([snap_df, df], ignore_index=True)
    out['repair_type'] = out['repair_type'].fillna('')
    out = out.sort_values([x_col, 'table_type', 'repair_type']).reset_index(drop=True)
    return out


def plot_one(ax, df, x_col, xlabel, include_snap, legend_loc):
    series = [('naive', '')] + [
        ('ivmh', ''),
        ('heap', 'Write Repair'),
        ('chain', 'Write Repair'),
        ('par', 'Write Repair'),
    ]
    for key in series:
        if not include_snap and key == ('naive', ''):
            continue
        label, color, linestyle, marker = STYLE[key]
        table_type, repair_type = key
        sub = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)].sort_values(x_col)
        if sub.empty:
            continue
        ax.plot(sub[x_col], sub['total_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Duration (ms / tx)')
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax.legend(loc=legend_loc, ncol=1, framealpha=0.95)


def render_pair(df, x_col, xlabel, stem):
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 4.1), sharey=False)
    plot_one(axes[0], df, x_col, xlabel, True, 'center left')
    plot_one(axes[1], df, x_col, xlabel, False, 'upper left')
    fig.tight_layout()
    out_pdf = FIGS_DIR / f'{stem}_{RUN_STAMP}.pdf'
    latest_pdf = FIGS_DIR / f'{stem}.pdf'
    fig.savefig(out_pdf, format='pdf')
    fig.savefig(latest_pdf, format='pdf')
    plt.show()
    print('Saved', out_pdf)
    print('Saved', latest_pdf)


def render_single(df, x_col, xlabel, stem, include_snap):
    fig, ax = plt.subplots(1, 1, figsize=(5.0, 4.1))
    legend_loc = 'center left' if include_snap else 'upper left'
    plot_one(ax, df, x_col, xlabel, include_snap, legend_loc)
    fig.tight_layout()
    out_pdf = FIGS_DIR / f'{stem}_{RUN_STAMP}.pdf'
    latest_pdf = FIGS_DIR / f'{stem}.pdf'
    fig.savefig(out_pdf, format='pdf')
    fig.savefig(latest_pdf, format='pdf')
    plt.show()
    print('Saved', out_pdf)
    print('Saved', latest_pdf)


In [ ]:
history_rows = []
for value in SWEEP['history_values']:
    print(f'Running history sweep at {value:.2%}')
    df = run_single(build_history_args(value), include_snap=CONFIG['rerun_snap'])
    df['history_ratio'] = value
    history_rows.append(df)
df_history = pd.concat(history_rows, ignore_index=True)
if not CONFIG['rerun_snap']:
    df_history = add_snap_rows(df_history, FIXED_SNAP_HISTORY_ROWS, 'history_ratio')
df_history['history_pct'] = df_history['history_ratio'] * 100.0
history_csv = DATA_DIR / f'sigmod_exp2_distinct_history_{CONFIG_TAG}_{RUN_STAMP}.csv'
history_latest = DATA_DIR / 'sigmod_exp2_distinct_history_latest.csv'
df_history.to_csv(history_csv, index=False)
df_history.to_csv(history_latest, index=False)
print('Saved', history_csv)
print('Saved', history_latest)
display(df_history.head())


In [ ]:
delta_rows = []
for value in SWEEP['delta_values']:
    print(f'Running delta sweep at {value:.2%}')
    df = run_single(build_delta_args(value), include_snap=CONFIG['rerun_snap'])
    df['delta_ratio'] = value
    delta_rows.append(df)
df_delta = pd.concat(delta_rows, ignore_index=True)
if not CONFIG['rerun_snap']:
    df_delta = add_snap_rows(df_delta, FIXED_SNAP_DELTA_ROWS, 'delta_ratio')
df_delta['delta_pct'] = df_delta['delta_ratio'] * 100.0
delta_csv = DATA_DIR / f'sigmod_exp2_distinct_delta_{CONFIG_TAG}_{RUN_STAMP}.csv'
delta_latest = DATA_DIR / 'sigmod_exp2_distinct_delta_latest.csv'
df_delta.to_csv(delta_csv, index=False)
df_delta.to_csv(delta_latest, index=False)
print('Saved', delta_csv)
print('Saved', delta_latest)
display(df_delta.head())


In [ ]:
render_pair(df_history, 'history_pct', 'Historical Scan Percentage (%)', 'exp2-distinct-history-pair')
render_pair(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'exp2-distinct-delta-pair')


In [ ]:
render_single(df_history, 'history_pct', 'Historical Scan Percentage (%)', 'exp2-distinct-history-with-snap', True)
render_single(df_history, 'history_pct', 'Historical Scan Percentage (%)', 'exp2-distinct-history-without-snap', False)
render_single(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'exp2-distinct-delta-with-snap', True)
render_single(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'exp2-distinct-delta-without-snap', False)
